In [ ]:
!apt-get install openjdk-17-jdk-headless -qq > /dev/null
!pip install pyspark gdown -qq

import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'

In [ ]:
import gdown

CATEGORY_NAME = 'Kindle_Store'
FILE_ID = '1AhlTkSclHtP8IL7LeON0FCcXRln7JvlH'
INPUT_PATH = f'/content/{CATEGORY_NAME}.jsonl.gz'

if not os.path.exists(INPUT_PATH):
    gdown.download(id=FILE_ID, output=INPUT_PATH, quiet=False)

OUTPUT_PATH = f'/content/outputs/fields_stat_{CATEGORY_NAME}.csv'

In [ ]:
from pyspark.sql import SparkSession
import json
import csv


def extract_fields(obj, prefix=''):
    fields = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            path = f'{prefix}.{k}' if prefix else k
            fields.append(path)
            fields.extend(extract_fields(v, path))
    elif isinstance(obj, list):
        for item in obj:
            fields.extend(extract_fields(item, prefix + '__list_of'))
    return fields


spark = SparkSession.builder \
    .appName('Fields Analysis') \
    .master('local[*]') \
    .config('spark.driver.memory', '40g') \
    .config('spark.python.worker.faulthandler.enabled', 'true') \
    .getOrCreate()

In [ ]:
sc = spark.sparkContext
rdd = sc.textFile(INPUT_PATH)

# Map phase
fields_rdd = rdd.map(json.loads).flatMap(lambda obj: extract_fields(obj)).map(lambda field: (field, 1))

# Reduce phase
field_counts = fields_rdd.reduceByKey(lambda a, b: a + b).sortBy(lambda x: (-x[1], x[0]))

# Save
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
rows = field_counts.collect()
with open(OUTPUT_PATH, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['field', 'count'])
    for k, v in rows:
        writer.writerow([k, v])

spark.stop()